### ***Autoecoder seleccionado***

In [ ]:
# -------- Dataloaders --------
train_dataloader = DataLoader(train_set, batch_size=100, shuffle = True)
valid_dataloader = DataLoader(valid_set, batch_size=100, shuffle = False)

# -------- Loss Function --------
loss_fn = nn.MSELoss()

# -------- Model --------
n=128
pre_trained_autoencoder_c = AutoencoderConvolutional(n=n, p1=0.2, p2=0.0, filter_num1=8, filter_num2=16, use_bn=False).to(device)

# -------- Optimizer --------
learning_rate = 1e-3
optimizer = torch.optim.Adam(pre_trained_autoencoder_c.parameters(), lr=learning_rate)

# -------- Stadistics --------
train_loss_mean = []
train_inc_loss_mean = []
valid_loss_mean = []

### ***Loop Clasificador***

In [ ]:
def valid_loop_clasiffier(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    valid_loss, valid_correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)

            # Statistics
            valid_loss += loss_fn(pred, y).item()
            valid_correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    valid_loss /= num_batches
    valid_correct /= size
    print(f"Test Error: \n Accuracy: {(100*valid_correct):>0.1f}%, Avg loss: {valid_loss:>8f} \n")

    return valid_loss, valid_correct

### ***Elección de Clasificador***

#### ***Baseline***

In [ ]:
# -------- Dataloaders --------
train_dataloader = DataLoader(train_set_orig, batch_size=100, shuffle = True)
valid_dataloader = DataLoader(valid_set_orig, batch_size=100, shuffle = False)

# -------- Loss Function --------
loss_fn = nn.CrossEntropyLoss()

# -------- Model --------
classifier_b = Classifier(autoencoder=pre_trained_autoencoder_c, use_bn=False).to(device)

# Freeze encoder and bottleneck
for param in classifier_b.encoder.parameters():
    param.requires_grad = False

for param in classifier_b.bottleneck.parameters():
    param.requires_grad = False

# -------- Optimizer --------
learning_rate = 1e-3
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, classifier_b.parameters()),
    lr=learning_rate
)

# DEBUG
# for name, p in classifier0.named_parameters():
#     print(name, p.requires_grad)

# -------- Statistics --------
train_loss_mean = []
train_inc_loss_mean = []
valid_loss_mean = []
train_acc_frac = []
valid_acc_frac = []

In [ ]:
EPOCHS = 40
for t in range(EPOCHS):
    print(f"Epoch {t+1}\n-------------------------------")
    train_inc_loss = train_loop(train_dataloader, classifier_b, loss_fn, optimizer)
    train_loss, train_acc = valid_loop_clasiffier(train_dataloader, classifier_b, loss_fn)
    valid_loss, valid_acc = valid_loop_clasiffier(valid_dataloader, classifier_b, loss_fn)

    train_inc_loss_mean.append(train_inc_loss)
    train_loss_mean.append(train_loss)
    valid_loss_mean.append(valid_loss)
    train_acc_frac.append(train_acc)
    valid_acc_frac.append(valid_acc)
print("Done!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

fig.suptitle("Setup baseline", fontsize=20)

# --- Gráfico 1: Cross Entropy Loss ---
axes[0].set_title("Cross Entropy Loss", fontsize=16)
axes[0].set_xlabel("Épocas", fontsize=12)
axes[0].set_ylabel("Error Promedio", fontsize=12)
axes[0].plot(range(EPOCHS), train_loss_mean, color='blue', label='Entrenamiento')
axes[0].plot(range(EPOCHS), valid_loss_mean, color='green', label='Validación')
# axes[0].plot(range(EPOCHS), train_inc_loss_mean, 'x:', color='tab:red', label='Entrenamiento Inc')
axes[0].legend()
axes[0].grid(True)

# --- Gráfico 2: Precisión ---
axes[1].set_title("Accuracy", fontsize=16)
axes[1].set_xlabel("Épocas", fontsize=12)
axes[1].set_ylabel("Precisión", fontsize=12)
axes[1].plot(range(EPOCHS), train_acc_frac, 'o-', c='crimson', label='Entrenamiento')
axes[1].plot(range(EPOCHS), valid_acc_frac, 'o-', c='olive', label='Validación')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

#### ***i) Nº de neuronas en capas ocultas***

In [ ]:
# -------- Dataloaders --------
train_dataloader = DataLoader(train_set_orig, batch_size=100, shuffle = True)
valid_dataloader = DataLoader(valid_set_orig, batch_size=100, shuffle = False)

# -------- Loss Function --------
loss_fn = nn.CrossEntropyLoss()

# -------- Model --------
classifier_i = Classifier(autoencoder=pre_trained_autoencoder_c, use_bn=False, n1= 512, n2=256).to(device)

# Freeze encoder and bottleneck
for param in classifier_i.encoder.parameters():
    param.requires_grad = False

for param in classifier_i.bottleneck.parameters():
    param.requires_grad = False

# -------- Optimizer --------
learning_rate = 1e-3
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, classifier_i.parameters()),
    lr=learning_rate
)

# DEBUG
# for name, p in classifier0.named_parameters():
#     print(name, p.requires_grad)

# -------- Statistics --------
train_loss_mean = []
train_inc_loss_mean = []
valid_loss_mean = []
train_acc_frac = []
valid_acc_frac = []

In [ ]:
EPOCHS = 40
for t in range(EPOCHS):
    print(f"Epoch {t+1}\n-------------------------------")
    train_inc_loss = train_loop(train_dataloader, classifier_i, loss_fn, optimizer)
    train_loss, train_acc = valid_loop_clasiffier(train_dataloader, classifier_i, loss_fn)
    valid_loss, valid_acc = valid_loop_clasiffier(valid_dataloader, classifier_i, loss_fn)

    train_inc_loss_mean.append(train_inc_loss)
    train_loss_mean.append(train_loss)
    valid_loss_mean.append(valid_loss)
    train_acc_frac.append(train_acc)
    valid_acc_frac.append(valid_acc)
print("Done!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

fig.suptitle("Setup i", fontsize=20)

# --- Gráfico 1: Cross Entropy Loss ---
axes[0].set_title("Cross Entropy Loss", fontsize=16)
axes[0].set_xlabel("Épocas", fontsize=12)
axes[0].set_ylabel("Error Promedio", fontsize=12)
axes[0].plot(range(EPOCHS), train_loss_mean, 's--', color='blue', label='Entrenamiento')
axes[0].plot(range(EPOCHS), valid_loss_mean, 's--', color='green', label='Validación')
# axes[0].plot(range(EPOCHS), train_inc_loss_mean, 'x:', color='tab:red', label='Entrenamiento Inc')
axes[0].legend()
axes[0].grid(True)

# --- Gráfico 2: Precisión ---
axes[1].set_title("Accuracy", fontsize=16)
axes[1].set_xlabel("Épocas", fontsize=12)
axes[1].set_ylabel("Precisión", fontsize=12)
axes[1].plot(range(EPOCHS), train_acc_frac, 'o-', c='crimson', label='Entrenamiento')
axes[1].plot(range(EPOCHS), valid_acc_frac, 'o-', c='olive', label='Validación')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

#### ***ii) Optimizador (a SGD)***

In [ ]:
# -------- Dataloaders --------
train_dataloader = DataLoader(train_set_orig, batch_size=100, shuffle = True)
valid_dataloader = DataLoader(valid_set_orig, batch_size=100, shuffle = False)

# -------- Loss Function --------
loss_fn = nn.CrossEntropyLoss()

# -------- Model --------
classifier_ii = Classifier(autoencoder=pre_trained_autoencoder_c, use_bn=False, n1= 512, n2=256).to(device)

# Freeze encoder and bottleneck
for param in classifier_ii.encoder.parameters():
    param.requires_grad = False

for param in classifier_ii.bottleneck.parameters():
    param.requires_grad = False

# -------- Optimizer --------
learning_rate = 1e-3
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, classifier_ii.parameters()),
    lr=learning_rate
)

# DEBUG
# for name, p in classifier0.named_parameters():
#     print(name, p.requires_grad)

# -------- Statistics --------
train_loss_mean = []
train_inc_loss_mean = []
valid_loss_mean = []
train_acc_frac = []
valid_acc_frac = []

In [ ]:
EPOCHS = 40
for t in range(EPOCHS):
    print(f"Epoch {t+1}\n-------------------------------")
    train_inc_loss = train_loop(train_dataloader, classifier_ii, loss_fn, optimizer)
    train_loss, train_acc = valid_loop_clasiffier(train_dataloader, classifier_ii, loss_fn)
    valid_loss, valid_acc = valid_loop_clasiffier(valid_dataloader, classifier_ii, loss_fn)

    train_inc_loss_mean.append(train_inc_loss)
    train_loss_mean.append(train_loss)
    valid_loss_mean.append(valid_loss)
    train_acc_frac.append(train_acc)
    valid_acc_frac.append(valid_acc)
print("Done!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

fig.suptitle("Setup 0", fontsize=20)

# --- Gráfico 1: Cross Entropy Loss ---
axes[0].set_title("Cross Entropy Loss", fontsize=16)
axes[0].set_xlabel("Épocas", fontsize=12)
axes[0].set_ylabel("Error Promedio", fontsize=12)
axes[0].plot(range(EPOCHS), train_loss_mean, 's--', color='blue', label='Entrenamiento')
axes[0].plot(range(EPOCHS), valid_loss_mean, 's--', color='green', label='Validación')
# axes[0].plot(range(EPOCHS), train_inc_loss_mean, 'x:', color='tab:red', label='Entrenamiento Inc')
axes[0].legend()
axes[0].grid(True)

# --- Gráfico 2: Precisión ---
axes[1].set_title("Accuracy", fontsize=16)
axes[1].set_xlabel("Épocas", fontsize=12)
axes[1].set_ylabel("Precisión", fontsize=12)
axes[1].plot(range(EPOCHS), train_acc_frac, 'o-', c='crimson', label='Entrenamiento')
axes[1].plot(range(EPOCHS), valid_acc_frac, 'o-', c='olive', label='Validación')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

#### ***iii) Dropout***

In [ ]:
# -------- Dataloaders --------
train_dataloader = DataLoader(train_set_orig, batch_size=100, shuffle = True)
valid_dataloader = DataLoader(valid_set_orig, batch_size=100, shuffle = False)

# -------- Loss Function --------
loss_fn = nn.CrossEntropyLoss()

# -------- Model --------
classifier_iii = Classifier(autoencoder=pre_trained_autoencoder_c, use_bn=False, n1= 512, n2=256).to(device)

# Freeze encoder and bottleneck
for param in classifier_iii.encoder.parameters():
    param.requires_grad = False

for param in classifier_iii.bottleneck.parameters():
    param.requires_grad = False

# -------- Optimizer --------
learning_rate = 1e-3
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, classifier_iii.parameters()),
    lr=learning_rate
)

# DEBUG
# for name, p in classifier0.named_parameters():
#     print(name, p.requires_grad)

# -------- Statistics --------
train_loss_mean = []
train_inc_loss_mean = []
valid_loss_mean = []
train_acc_frac = []
valid_acc_frac = []

In [ ]:
EPOCHS = 40
for t in range(EPOCHS):
    print(f"Epoch {t+1}\n-------------------------------")
    train_inc_loss = train_loop(train_dataloader, classifier_iii, loss_fn, optimizer)
    train_loss, train_acc = valid_loop_clasiffier(train_dataloader, classifier_iii, loss_fn)
    valid_loss, valid_acc = valid_loop_clasiffier(valid_dataloader, classifier_iii, loss_fn)

    train_inc_loss_mean.append(train_inc_loss)
    train_loss_mean.append(train_loss)
    valid_loss_mean.append(valid_loss)
    train_acc_frac.append(train_acc)
    valid_acc_frac.append(valid_acc)
print("Done!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

fig.suptitle("Setup 0", fontsize=20)

# --- Gráfico 1: Cross Entropy Loss ---
axes[0].set_title("Cross Entropy Loss", fontsize=16)
axes[0].set_xlabel("Épocas", fontsize=12)
axes[0].set_ylabel("Error Promedio", fontsize=12)
axes[0].plot(range(EPOCHS), train_loss_mean, 's--', color='blue', label='Entrenamiento')
axes[0].plot(range(EPOCHS), valid_loss_mean, 's--', color='green', label='Validación')
# axes[0].plot(range(EPOCHS), train_inc_loss_mean, 'x:', color='tab:red', label='Entrenamiento Inc')
axes[0].legend()
axes[0].grid(True)

# --- Gráfico 2: Precisión ---
axes[1].set_title("Accuracy", fontsize=16)
axes[1].set_xlabel("Épocas", fontsize=12)
axes[1].set_ylabel("Precisión", fontsize=12)
axes[1].plot(range(EPOCHS), train_acc_frac, 'o-', c='crimson', label='Entrenamiento')
axes[1].plot(range(EPOCHS), valid_acc_frac, 'o-', c='olive', label='Validación')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()